## **TP02: CleaningData - Applying OOP and Design Patterns**

#### Importing Libraries

In [1]:
from abc import ABC, abstractmethod
import time 
import pandas as pd

### Exercise 1: Build the CSVReader Class (OOP Foundation) 

**Objective:**

- Reinforce OOP fundamentals: classes, constructors, methods, encapsulation. 

- Learn to design reusable data classes for DS projects. 

**Concept:**

A CSVReader class should encapsulate all functionality related to reading CSV files. 

**Instructions:**

1. Create a class CSVReader. 

2. Define an __init__ method that takes file_path. 

3. Add a read() method that loads the CSV file using pandas. 

4. Add a preview(n) method that prints the top n rows. 

In [2]:
class CSVReader:
    def __init__(self, file_path: str):
        self.file_path = file_path
        self.df = None

    def read(self) -> pd.DataFrame:
        self.df = pd.read_csv(self.file_path)
        return self.df

    def preview(self, n: int =5) -> pd.DataFrame:
        if self.df is not None:
            return self.df.head(n)
        print("Dataset no loaded yet.")
        return pd.DataFrame()

### Exercise 2: Apply Pattern for Missing Values Handling 

**Objective:** 
- Learn the Strategy Pattern - one interface, many interchangeable behaviors. 

- Practice inheritance and polymorphism. 

**Concept:**

We often need different strategies to handle missing data (drop, fill mean, fill mode, etc.). 
The pattern allows flexible switching between methods without changing the main code. 

**Instructions:** 

1. Create an abstract class MissingValueStrategy with a method handle(df). 

2. Create subclasses: 
    
    - DropMissing 

    - FillMean 
    
    - FillMode 

3. Create a DataCleaner class that applies the chosen strategy.

In [3]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

class DropMissing(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.dropna()

class FillMean(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.fillna(df.select_dtypes(include="number").mean())

class FillMode(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.fillna(df.mode().iloc[0])

### Exercise 3: Add Decorators for Logging and Timing

**Objective:**

- Learn the Decorator Pattern to add reusable functionality (logging, timing).

- Understand how decorators support the “Open/Closed Principle”.

**Concept:**

- Decorators wrap functions to extend their behavior without altering the original code.

**Instructions:**

1. Create two decorators:

    - @log_action → logs when a method starts and finishes.

    - @log_time → calculates and prints execution time.

2. Apply them to methods in CSVReader or DataCleaner.

In [4]:
def log_action(func):
    def wrapper(*args, **kwargs):
        print(f"[LOG] Starting: {func.__name__}")
        result = func(*args, **kwargs)
        print(f"[LOG] Finished: {func.__name__}")
        return result
    return wrapper

def log_time(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        elapsed_time = time.time() - start_time
        print(f"[TIME] {func.__name__} excecuted in {elapsed_time:.4f} seconds")
        return result
    return wrapper

### Exercise 4: Implement Factory Pattern for Data Transformations

**Objective:**

- Practice Factory Pattern for scalable creation of transformation objects.

- Apply abstraction and composition.

**Concept:**

Instead of manually creating objects, use a factory that decides what transformation to apply.

**Instructions:**

1. Create an abstract class DataTransform with a method apply(df).

2. Implement subclasses:

    - NormalizeColumns

    - RemoveDuplicates

    - StandardizeText

3. Create a TransformFactory that returns transformation objects based on string input.


In [5]:
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

class NormalizeColumns(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
        return df

class RemoveDuplicates(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates()

class StandardizeText(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        for col in df.select_dtypes(include=["object"]):
            df[col] = df[col].astype(str).str.strip().str.title()
        return df

class TransformFactory:
    @staticmethod
    def create_transform(transform_type: str) -> DataTransform:
        transformers = {
            "normalize": NormalizeColumns(),
            "duplicates": RemoveDuplicates(),
            "text": StandardizeText()
        }
        if transform_type not in transformers:
            raise ValueError(f"Unknown transform type: {transform_type}")
        return transformers[transform_type]

### Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

**Objective:**

- Combine all previous concepts into one cleaning pipeline.

- Use the Template Method Pattern to define a consistent workflow.

**Concept:**

- The Template Method defines the skeleton of a process and lets subclasses override specific steps.

**Instructions:**

1. Create an abstract class DataPipeline with a run() method defining steps:

    - load()

    - clean()

    - transform()
    - save()

2. Create CSVDataPipeline that implements each step using previous classes.


In [6]:
class DataPipeline(ABC):
    """
    The Template Method defines the skeleton of a data processing algorithm.
    """
    def run(self, output_path: str = None) -> pd.DataFrame:
        df = self.load()
        df = self.clean(df)
        df = self.transform(df)
        if output_path:
            self.save(df, output_path)
        return df

    @abstractmethod
    def load(self) -> pd.DataFrame:
        pass

    @abstractmethod
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

    @abstractmethod
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

    @abstractmethod
    def save(self, df: pd.DataFrame, output_path: str):
        pass


class CSVDataPipeline(DataPipeline):
    def __init__(self, file_path: str, missing_strategy: MissingValueStrategy, transform_types: list):
        self.reader = CSVReader(file_path)
        self.missing_strategy = missing_strategy
        self.transform_types = transform_types

    @log_action
    @log_time
    def load(self) -> pd.DataFrame:
        return self.reader.read()

    @log_action
    @log_time
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.missing_strategy.handle(df)

    @log_action
    @log_time
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        for t_type in self.transform_types:
            transformer = TransformFactory.create_transform(t_type)
            df = transformer.apply(df)
        return df

    @log_action
    @log_time
    def save(self, df: pd.DataFrame, output_path: str):
        df.to_csv(output_path, index=False)
        print(f"Cleaned data successfully saved to {output_path}")


# Example case
if __name__ == "__main__":
    # Instantiate the pipeline with a strategy and list of transformations
    pipeline = CSVDataPipeline(
        file_path="sample_data.csv",
        missing_strategy=FillMean(),
        transform_types=["normalize", "duplicates", "text"]
    )
    
    # Run the full pipeline
    cleaned_df = pipeline.run(output_path="cleaned_data.csv")

[LOG] Starting: wrapper
[TIME] load excecuted in 0.0403 seconds
[LOG] Finished: wrapper
[LOG] Starting: wrapper
[TIME] clean excecuted in 0.0130 seconds
[LOG] Finished: wrapper
[LOG] Starting: wrapper
[TIME] transform excecuted in 0.0150 seconds
[LOG] Finished: wrapper
[LOG] Starting: wrapper
Cleaned data successfully saved to cleaned_data.csv
[TIME] save excecuted in 0.0050 seconds
[LOG] Finished: wrapper
